In [6]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torch.utils.data import DataLoader, TensorDataset

import torch.optim as optim
from sklearn.model_selection import train_test_split
import mlflow
from mlflow.models.signature import infer_signature

from tqdm import tqdm

from faiss import write_index, read_index

import warnings
warnings.filterwarnings('ignore')

In [7]:
pd.set_option('display.max_colwidth', None)


In [7]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [8]:
files = ['product_brand_embedding.npy', 'product_bullet_point_embedding.npy', 'product_color_embedding.npy', 'product_title_embedding.npy', 'product_description_embedding.npy', 'query_embedding.npy']

data = []
for file in files:
    data.append(np.load(f'../data/new_embeddings/{file}'))

In [9]:
embedding = np.concat((data[0],data[1],data[2],data[3],data[4],data[5]),axis=1)

In [10]:
labels = pd.read_csv(f'../data/data.csv')['binary_label'].values

In [11]:
embedding.shape

(99909, 4608)

In [12]:
labels.shape

(99909,)

In [13]:
x_train, x_test, y_train, y_test = train_test_split(embedding, labels, test_size=0.2, random_state=42, shuffle=True)

In [14]:
x_train = torch.from_numpy(x_train)
x_test = torch.from_numpy(x_test)
y_train = torch.from_numpy(y_train)
y_test = torch.from_numpy(y_test)

In [15]:
product_idx = 3840

In [16]:
# query_embeddings = x_train[:79904,product_idx:]
# product_embeddings = x_train[:79904,:product_idx]

In [17]:
torch.rand(768,32).reshape(32,768).shape

torch.Size([32, 768])

In [18]:
class QueryTower(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(QueryTower, self).__init__()
        self.query_tower = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim)
        )
    
    def forward(self, query_input):
        query_input = query_input.float()
        query_input = query_input.view(-1,768)
        query_emb = self.query_tower(query_input)
        query_emb = nn.functional.normalize(query_emb, p=2, dim=1)
        
        return query_emb


In [19]:
class ProductTower(nn.Module):    
    def __init__(self, input_dim, output_dim):
        super(ProductTower, self).__init__()
        self.product_tower = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim)
        )
    
    def forward(self, product_input):
        product_input = product_input.float()
        product_input = product_input.view(-1,3840)
        product_emb = self.product_tower(product_input)
        product_emb = nn.functional.normalize(product_emb, p=2, dim=1)
    
        return product_emb
    

In [20]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, query_emb, pos_product_emb, neg_product_emb):
        pos_dist = torch.norm(query_emb - pos_product_emb, p=2, dim=1)
        neg_dist = torch.norm(query_emb - neg_product_emb, p=2, dim=1)
        # print(pos_dist.shape, neg_dist.shape)
        loss = torch.mean(torch.relu(pos_dist - neg_dist + self.margin))
        # print(loss)
        return loss

In [21]:
query_dim = 768
product_dim = 3840

output_dim = 32
batch_size = 32

train_size = x_train.shape[0] - (x_train.shape[0] % 32)
num_epochs = 100
test_size = x_test.shape[0] - (x_test.shape[0] % 32)

train_num_batches = int(train_size/batch_size)
test_num_batches = int(test_size/batch_size)

query_model = QueryTower(query_dim, output_dim).to(device)
product_model = ProductTower(product_dim, output_dim).to(device)
optimizer = optim.Adam(list(query_model.parameters()) + list(product_model.parameters()), lr=0.001)
criterion = ContrastiveLoss()

train_dataset = TensorDataset(x_train[:train_size,product_idx:], x_train[:train_size,:product_idx])
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = TensorDataset(x_test[:test_size,product_idx:], x_test[:test_size,:product_idx])
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

In [18]:
train_size

79904

In [22]:
mlflow.set_tracking_uri(uri='http://127.0.0.1:5000')
mlflow.start_run()
mlflow.log_param("epochs", num_epochs)

loss = 0

# Training loop
for epoch in tqdm(range(num_epochs)):    

    query_model.train()
    product_model.train()
    train_batch_loss = 0

    for query_batch, pos_product_batch in train_dataloader:

        query_batch = query_batch.to(device)
        pos_product_batch = pos_product_batch.to(device)

        # Negative sampling: shuffle product embeddings
        neg_product_batch = x_train[:train_size,:product_idx][torch.randperm(train_size)[:batch_size]]
        neg_product_batch = neg_product_batch.to(device)

        # print(query_batch.shape, pos_product_batch.shape, neg_product_batch.shape)

        # Forward pass
        query_emb = query_model(query_batch)
        pos_product_emb = product_model(pos_product_batch)
        neg_product_emb = product_model(neg_product_batch)

        # print(query_emb.shape, pos_product_emb.shape, neg_product_emb.shape)
        # Compute loss
        train_batch_loss = criterion(query_emb, pos_product_emb, neg_product_emb)

        # Backpropagation
        optimizer.zero_grad()
        train_batch_loss.backward()
        optimizer.step()

        train_batch_loss += train_batch_loss

    total_train_loss = train_batch_loss/train_num_batches

    query_model.eval()
    product_model.eval()

    test_batch_loss = 0
    with torch.no_grad():
        for query_batch, product_batch in test_dataloader:

            query_batch = query_batch.to(device)
            product_batch = product_batch.to(device)


            neg_product_batch = x_test[:test_size,:product_idx][torch.randperm(test_size)[:batch_size]]
            neg_product_batch = neg_product_batch.to(device)
            
            # print(query_batch.shape, product_batch.shape, neg_product_batch.shape)
            query_emb = query_model(query_batch)
            pos_product_emb = product_model(pos_product_batch)
            neg_product_emb = product_model(neg_product_batch)

            test_batch_loss = criterion(query_emb, pos_product_emb, neg_product_emb)

            test_batch_loss += test_batch_loss

    total_test_loss = test_batch_loss/test_num_batches

    if (epoch+1) % 10 == 0:    
        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {total_train_loss:.4f}, Test Loss: {total_test_loss:.4f}")


mlflow.log_metric("Average Loss", total_test_loss)
mlflow.pytorch.log_model(product_model, 'Product Tower',registered_model_name='product_tower')
mlflow.pytorch.log_model(query_model, 'Query Tower',registered_model_name='query_tower')

mlflow.end_run()


 10%|█         | 10/100 [02:06<18:36, 12.40s/it]

Epoch 10/100, Train Loss: 0.0001, Test Loss: 0.0032


 20%|██        | 20/100 [04:02<14:25, 10.82s/it]

Epoch 20/100, Train Loss: 0.0001, Test Loss: 0.0030


 30%|███       | 30/100 [06:13<15:33, 13.34s/it]

...

100%|██████████| 100/100 [21:07<00:00, 12.68s/it]

Epoch 100/100, Train Loss: 0.0000, Test Loss: 0.0033



2025/04/04 11:07:29 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'product_tower' already exists. Creating a new version of this model...
2025/04/04 11:07:29 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: product_tower, version 2
Created version '2' of model 'product_tower'.
2025/04/04 11:07:32 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'query_tower' already exists. Creating a new version of this model...
2025/04/04 11:07:32 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: query_tower, version 10


🏃 View run nimble-gnat-824 at: http://127.0.0.1:5000/#/experiments/0/runs/4b210c961b6a4d5996219ea5e5dfc64f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0


Created version '10' of model 'query_tower'.


In [23]:
embedding = torch.from_numpy(embedding)

In [24]:
import faiss

product_model.eval()
with torch.no_grad():
    product_embeddings_faiss = product_model(embedding[:,:3840].to(device))


In [25]:
index = faiss.IndexFlatL2(32)

index.add(product_embeddings_faiss.cpu().numpy())

In [ ]:

write_index(index, "../assets/product_tower_embedding.index")

In [6]:
index = read_index("../assets/product_tower_embedding.index")

In [37]:
def retrieve_products(query, top_k=5):
    query_emb = query_model(query.to(device))
    query_emb_np = query_emb.cpu().detach().numpy()
    
    _, indices = index.search(query_emb_np, top_k) 
    return indices

In [11]:
df = pd.read_csv('../data/data.csv')

In [10]:
df.columns

Index(['product_id', 'product_title', 'product_description',
       'product_bullet_point', 'product_brand', 'product_color', 'query',
       'esci_label', 'split', 'binary_label'],
      dtype='object')

In [14]:
df.iloc[848]

product_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         B071LN6Q9Y
product_title                                                                                                                                                                                                                                                                                                                                                                                                                                          hangman products elephant hook ceiling hanger  whit

In [3]:
from sentence_transformers import SentenceTransformer

emb_model = SentenceTransformer('../assets/gte-multilingual-base',trust_remote_code=True)


Some weights of the model checkpoint at ../assets/gte-multilingual-base were not used when initializing NewModel: {'classifier.weight', 'classifier.bias'}
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [7]:
def get_embeddings(text):
    embeddings = emb_model.encode(text, normalize_embeddings=False, show_progress_bar=True)
    return embeddings.squeeze()


In [29]:
emb_example = torch.from_numpy(get_embeddings(df.iloc[100].to_list()[1:7]))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [30]:
emb_example = emb_example.reshape((1,4608))

In [ ]:
new_query = emb_example[:,product_idx:]
product = emb_example[:,:product_idx]

In [ ]:
recommended_products = retrieve_products(new_query, top_k=5)
print("Recommended Product Indices:", recommended_products)

Recommended Product Indices: [[28094   100 51588 23565 50833]]


In [39]:
recommended_products[0]

array([28094,   100, 51588, 23565, 50833])

In [40]:
df.iloc[100]

product_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [41]:
df.iloc[recommended_products[0]]

,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,query,esci_label,split,binary_label
28094,B089GYKNPL,2pcs fashion couple rings stainless ring steel rotatable bottle opener party ring creative versatile beer bottle openershipment from usa fast delivery blakslive 7,punk wedding mens stainless band ring spin titanium steel rotatable chain product description this is a cool chaintype rotating ring that can be used as a bottle opener you can perform magic tricks to open the beer smoothly at the party this is very enviable and very interesting but also makes us a party star stylish design exquisite fashion technical index material perfect titanium steel product weight 6g note due to the different display and lighting effects the actual color of the product may be slightly different from the color shown on the picture,fashion beer bottle openereasy to open the bottle it can make you in the party the field the bar the school and other gatherings with friends make you quick and quick beer bottle so you quickly become a friends eye magician give your friends a deep impression\nmade of highquality stainless steel durable the inside and surface are well polished and comfortable to wear it can withstand prolonged wear and tear\nthis ring has an inner ring that can be rotated manually and locked in the main chain this is a very interesting ring a stylish design suitable for everyday use and very suitable as a tool to relieve stress especially for manic or irritable people\nunique and fashionable designa meaningful way to share lovevery suitable for the gift selection of special occasions such as christmas valentines day birthday anniversary wedding very suitable for loversyou can also think of it as a friendship ring one for you and the other for your friends,na,blakslive,bottle opener ring,E,train,1
100,B07V35Y7M2,10 pieces ring bottle opener stainless steel beer bottle opener colorful finger bottle opener for party family gift supplies,features practical size every beautiful and practical ring bottle opener is only 22 mm 087 inches it fits on the finger very well which makes it easier to open the bottle portable and convenient to carry sturdy material these ring bottle openers are made of quality stainless steel and they are extremely sturdy and durable they help you open every bottle faster and safer can last for a long time use stylish design all ring bottle openers are designed to be very stylish keeping up with the trend of the times bright colors will make them more popular with everyone bring you good mood when using it specifications material stainless steel color as pictures shown size 22 mm 087 inches package includes 10 x ring bottle openers,practical size every beautiful and practical ring bottle opener is only 22 mm 087 inches it fits on the finger very well which makes it easier to open the bottle portable and convenient to carry\nsturdy material these ring bottle openers are made of quality stainless steel and they are extremely sturdy and durable they help you open every bottle faster and safer can last for a long time use\nrich variety there are five different colors ring bottle openers in each bag and each comes with two colors there are a wide variety of choices for you to choose from which is definitely enough for your daily life color as pictures shown\ndiverse usage you can use these ring bottle openers to open the bottles at the party or you can give them as a gift to your family or friends to better promote your relationship it will definitely be a very popular gift\nstylish design all ring bottle openers are designed to be very stylish keeping up with the trend of the times bright colors will make them more popular with everyone bring you good mood when using it,boao,as pictures shown,bottle opener ring,E,train,1
51588,B074FXTH6K,vqysko bottle opener ring bandbeer bar tool creative versatile stainless steel finger party ring silver 8,vqysko beer bar tool creative versa

In [22]:
torch.rand(1,768).view(-1,768).shape

torch.Size([1, 768])

# query tower inference

In [3]:
import requests
import torch

url = "http://localhost:5001/invocations"

input_tensor = torch.rand(1,768)

# Convert to list
data = {"inputs": input_tensor.tolist()}

# Send request
response = requests.post(url, json=data)

# Print prediction
print(response.json())


{'predictions': [[-0.12178698182106018, 0.3475428819656372, -0.0008688937523402274, 0.003265833016484976, 0.17057190835475922, 0.19004526734352112, -0.13434579968452454, -0.1344868391752243, -0.019446058198809624, 0.05593753978610039, 0.11845600605010986, 0.35243669152259827, -0.08282971382141113, 0.14524118602275848, -0.348490446805954, 0.0861508697271347, -0.03892110288143158, -0.16809740662574768, 0.0006604751106351614, -0.08071773499250412, -0.032925426959991455, -0.35876038670539856, -0.17257872223854065, 0.2554073929786682, -0.08478299528360367, -0.07180222868919373, 0.04382268339395523, 0.34291860461235046, -0.03623281419277191, -0.1698543280363083, 0.023454593494534492, -0.20194444060325623]]}


In [4]:
response.json()['predictions']

[[-0.12178698182106018,
  0.3475428819656372,
  -0.0008688937523402274,
  0.003265833016484976,
  0.17057190835475922,
  0.19004526734352112,
  -0.13434579968452454,
  -0.1344868391752243,
  -0.019446058198809624,
  0.05593753978610039,
  0.11845600605010986,
  0.35243669152259827,
  -0.08282971382141113,
  0.14524118602275848,
  -0.348490446805954,
  0.0861508697271347,
  -0.03892110288143158,
  -0.16809740662574768,
  0.0006604751106351614,
  -0.08071773499250412,
  -0.032925426959991455,
  -0.35876038670539856,
  -0.17257872223854065,
  0.2554073929786682,
  -0.08478299528360367,
  -0.07180222868919373,
  0.04382268339395523,
  0.34291860461235046,
  -0.03623281419277191,
  -0.1698543280363083,
  0.023454593494534492,
  -0.20194444060325623]]

# embedding model inference

In [16]:
import requests
import torch

url = "http://localhost:5001/invocations"

input_tensor = torch.rand(1,768)

# Convert to list
data = {"inputs": ['bottle opener ring'],
        }

# Send request
response = requests.post(url, json=data)

# Print prediction
print(response.json())


{'predictions': [[-0.04666038602590561, 0.05391041934490204, 0.020811129361391068, 0.0009996165754273534, 0.1019439697265625, -0.04832988232374191, -0.02423499897122383, -0.019809257239103317, 0.07352948933839798, -0.017443235963582993, -0.06511249393224716, 0.01288723573088646, -0.07214204221963882, 0.03678182512521744, -0.07344791293144226, 0.060947973281145096, 0.03496871516108513, -0.03130273148417473, 0.042264603078365326, 0.07275903224945068, 0.08825119584798813, 0.029291311278939247, 0.05080761760473251, -0.07254785299301147, -0.04930812120437622, 0.10655299574136734, 0.020048774778842926, -0.043175939470529556, -0.04609251767396927, -0.023546215146780014, -0.02543213590979576, -0.0632701888680458, -0.03787185251712799, -0.010317663662135601, 0.10689179599285126, 0.030998487025499344, 0.022398168221116066, 0.02829151600599289, -0.04349838197231293, 0.034068141132593155, -0.0342831127345562, -0.016605820506811142, 0.0422082357108593, -0.01491609774529934, -0.04781549423933029, 0.

In [4]:
! pwd

/home/linuxachin/Desktop/Codes/Two-Tower-Recommendation/notebook


In [2]:
import sys
import os

# Add the project directory (parent of notebook and scripts) to sys.path
project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_path not in sys.path:
    sys.path.append(project_path)

# # Now you can import modules from scripts
# import scripts.my_script  # Import a specific script


In [9]:
from scripts import get_embedding, get_tower_embedding, index_search
# import get_embedding

query = "bottle opener"

# print(name)

query_tower_url = "http://localhost:5001/invocations"
embedding_model_url = "http://localhost:5002/invocations"


q_embedding = get_embedding.get_embedding_from_model(query,embedding_model_url)
q_tower_embedding = get_tower_embedding.get_query_tower_embedding(q_embedding,query_tower_url)
recommendations = index_search.retrieve_products(q_tower_embedding,index_path='../assets/product_tower_embedding.index',top_k=10)


In [10]:
recommendations

,product_title,product_description,product_brand
47456,gacube bartender bottle openersstainless steel heavy duty beer bottle openers orange,gacube bartender bottle openersstainless steel heavy duty beer bottle openers orange,gacube
82428,rommeka bottle can and jar grip opener multi function 4in1 5in1 and 6in1 kitchen tools set anti slip lid seal remover easy twist off for children elderly arthritic and weak hand pack of 3,rommeka bottle can and jar opener you dont need to ask for help with this convenient kitchen gadgets and save your hands from the pain and stress these multi opening tools are practical and lightweight can be carry to anywhere also very easy to use idea for family gathering colleagues gathering bbq picnic and so on best gift for family or friends 4in1 jarlid opener 4 different size opener allows you gauge any lid or cap size choose appropriate opener size and twist off slipping or breakage 5in1 bottle opener popular handy device for safe opening cans with ring release vacuum on jar lids open any can jar or bottle with minimum effort this tool is great for opening those water bottles that have short tops that are hard to grip easily twist off screw caps with a simple twist 6in1 sode can opener easy tearing pulling turning and twisting for anything you crave open pop tabs pull tabs bags safety seals metal and plastic bottle caps jars and more with to no efforts tips please avoid exposing the jar opener to direct sunlight for fear the rubber aging package included 1 x 4in1 jar opener 1 x 5in1 bottle opener 1 x 6in1 can opener,rommeka
47359,ahumans bottle opener hammer of thor shaped bottle opener abs marvel beer opener golden silvery gold,ahumans bottle opener hammer of thor shaped bottle opener abs marvel beer opener gold silvery specification color golden silvery size 1657047mm649276185 inches material abs metal usage scenarios home bars restaurants hotels party venue bbq package included 1bottle opener customer service our customer service team will try their best to help all the buyers deal with various kinds of issues in 24 hours please contact them first via seller platform email if you have any questions,ahumans
92658,3 packs drincarier waiters corkscrew professional wine openersupgraded heavy duty stainless steel wine keyclassic allinone corkscrew bottle opener and foil cutter 3pack pink,drincarier teflon waiters corkscrew is a classic corkscrew design that has a foil cutter blade a two stage cork lifter and bottle opener open a wine bottle by twisting in the screw placing the openers shortest step on the top of the bottle lip and pivoting up the handle after raising the cork partially lower the handle to engage the longest opener step on the bottle lip then pivot on the handle to remove the cork from the bottlesmall and easy to carry after folded you can take it anywhere for enjoying wine when campinghiking or travelingperfect for serverssommeliersbartenders and daily needs in home,drincarier
34959,key bottle openers assorted vintage skeleton keys wedding party favors pack of 70 silver,package include 70 pieces vintage key bottle openers assorted antique silver skeleton keys 10 pieces each style 7 styles quality design look and feel each is made of sturdy quality metal alloy and has an antiqued color finish to give them a beautiful vintage look the keys measure between 275 35 inches long,xonor
31295,nidavellir 2pack mjolnir keychain bottle opener thor hammer bottle opener keychain thor keychain beer bottle opener silver bronze,thors hammer keychain bottle opener forged in the heart of a dying star its power has no equal as a weapon to destroy or as a tool to build it is fit companion for a king but im just used to open the beer excellent features thor keychains made of highquality zinc alloy and double polished for a shiny rustproof and breakresistant surface cool beer bottle opener keychain with exquisite workmanship looks terrific and high end great novelty for using in parties creative and stunning d